# 03 · EVE — an unsupervised evolutionary model

**CFTR variant-effect toolkit · beginner track.** EVE (*Evolutionary model of Variant Effect*, Frazer et al. 2021, *Nature*, PMID 34707284) is a deep generative model of a protein family's evolutionary variation. It is **unsupervised** — it never saw a clinical label — which is what makes it fair to benchmark against ClinVar later (notebook 13).

> ⚠️ **DEMO DATA.** The EVE scores here are hand-authored illustrative values for ~13 curated CFTR variants — **not** real EVE output. Every table keeps a `source` column reading `DEMO`. See the *How to get REAL data* box, then join real per-variant scores on `protein_variant`; the code runs unchanged once `source` is `REAL`.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import toolkit as tk
import pandas as pd, numpy as np
%matplotlib inline

## 1 · EVE — an evolutionary generative model

**What is it?** EVE (*Evolutionary model of Variant Effect*) is a deep **generative model** trained on the **multiple-sequence alignment (MSA)** of a protein family — i.e. the same protein lined up across hundreds or thousands of species. Evolution has already run a giant natural experiment: positions that *must not change* to keep the protein working stay constant across species, while positions that tolerate change vary freely.

EVE learns that pattern of *evolutionary constraint*, then scores a new variant by asking: **"how well does this amino-acid change fit what evolution has tolerated here?"** A change at a rigidly-conserved position looks evolutionarily *implausible* → high score.

- **Score range:** `[0, 1]` (a posterior probability of being pathogenic).
- **Rule of thumb:** `>= 0.5` ~ pathogenic, `< 0.5` ~ benign. Higher = more damaging.
- **Unsupervised:** trained only on sequences, *never* on ClinVar labels → **low circularity** when we later benchmark it against clinical databases.

> ### 📥 How to get REAL EVE data
> Download the per-protein CSV for **CFTR (UniProt `P13569`)** from **https://evemodel.org**, then join it onto your variants by `protein_variant` (e.g. `G551D`). Replace `tk.load_eve()` below with your merged, real table.

In [2]:
# EVE demo scores. Note the `source` column: every row says DEMO.
eve = tk.load_eve()
print(f'{len(eve)} variants | columns: {list(eve.columns)}')
eve

13 variants | columns: ['protein_variant', 'eve_score', 'source']


,protein_variant,eve_score,source
0,G551D,0.980,DEMO
2,R117H,0.420,DEMO
3,R334W,0.900,DEMO
4,G85E,0.930,DEMO
5,D1152H,0.610,DEMO
6,R668C,0.180,DEMO
7,Tyr161Cys,0.832,DEMO
8,Gly970Asp,0.773,DEMO
9,Ser912Leu,0.742,DEMO
10,Val520Phe,0.718,DEMO


## 2 · Turn an EVE score into a call

`tk.call_from_score(score, 'eve')` applies EVE's published threshold (`>= 0.5` ~ pathogenic, higher = worse) and returns `pathogenic / uncertain / benign`.

In [3]:
eve['eve_call'] = eve['eve_score'].apply(lambda s: tk.call_from_score(s, 'eve'))
eve[['protein_variant', 'eve_score', 'eve_call', 'source']]

,protein_variant,eve_score,eve_call,source
0,G551D,0.980,pathogenic,DEMO
2,R117H,0.420,benign,DEMO
3,R334W,0.900,pathogenic,DEMO
4,G85E,0.930,pathogenic,DEMO
5,D1152H,0.610,pathogenic,DEMO
6,R668C,0.180,benign,DEMO
7,Tyr161Cys,0.832,pathogenic,DEMO
8,Gly970Asp,0.773,pathogenic,DEMO
9,Ser912Leu,0.742,pathogenic,DEMO
10,Val520Phe,0.718,pathogenic,DEMO


## Key takeaways

1. **EVE** models a protein family's **evolutionary** constraint (its MSA). Score `[0,1]`, `>= 0.5` ~ pathogenic, **higher = worse**.
2. It is **unsupervised** w.r.t. clinical labels → **low circularity** vs ClinVar.
3. Everything here is **DEMO** — swap in the real per-protein CSV (evemodel.org, UniProt P13569), join on `protein_variant`, and every cell runs unchanged.

**Next:** notebook 04 — **ESM1b**, a protein language model whose scale runs *backwards*. EVE-vs-ESM1b comparison lives in the integration notebook (12).